# Extra runs for Shuttle, Yeast and Ecoli (Colab)
Run the cells **one by one, from top to bottom**. Wait for each cell to finish before the next one. If this Colab session was used before, first click **Runtime > Restart session**.

**Why:** the first six datasets already have the network, attribute-level, alpha-sensitivity and gate-comparison results. Shuttle, Yeast and Ecoli do not. This notebook runs exactly the same code on those three, so every table in the paper can cover all nine datasets.

**You need no files from your computer.** Total run time is about 1 to 1.5 hours. Each cell saves its results after every dataset, so if Colab disconnects you only repeat the unfinished dataset.

## Cell 1: get your repo and one package

In [ ]:
%cd /content
!rm -rf conformal-fuzzy-ccl
!git clone -q https://github.com/mahibgm9676-lab/conformal-fuzzy-ccl.git
%cd /content/conformal-fuzzy-ccl
!pip -q install ucimlrepo
!ls ccl_helpers.py

## Cell 2: download the three datasets (same rule as before: classes with fewer than 8 cases are dropped)

In [ ]:
import os, pandas as pd
from ucimlrepo import fetch_ucirepo
os.makedirs('data', exist_ok=True)
for name, i in {'shuttle': 148, 'yeast': 110, 'ecoli': 39}.items():
    d = fetch_ucirepo(id=i)
    Xf = d.data.features.select_dtypes('number')
    yt = d.data.targets.iloc[:, 0].astype(str)
    out = pd.concat([Xf, yt.rename('class')], axis=1)
    cnt = out['class'].value_counts()
    drop = cnt[cnt < 8].index.tolist()
    out = out[~out['class'].isin(drop)]
    out.to_csv(f'data/{name}.csv', index=False)
    print(name, out.shape, 'dropped:', drop, '| class sizes:', out['class'].value_counts().tolist())

## Cell 3: check that these are the same data as before
The instances must be **58000** (shuttle), **1479** (yeast) and **327** (ecoli). The UCI pages list 7, 8 and 7 features. Please send me what this cell prints, so the dataset table in the paper is exact.

In [ ]:
import numpy as np
import ccl_helpers as H
paths = {'shuttle': 'data/shuttle.csv', 'yeast': 'data/yeast.csv', 'ecoli': 'data/ecoli.csv'}
for n, p in paths.items():
    X, y = H.load(p)
    print(n, 'instances', X.shape[0], '| features', X.shape[1], '| classes', len(np.unique(y)), '| class sizes', sorted(np.bincount(y).tolist()))

## Cell 4: part A, 20 splits (alpha sensitivity, concept scores, and the gate comparison). About 15 to 25 minutes

In [ ]:
import os, time, pandas as pd
t0 = time.time()
for n, p in paths.items():
    f_cp, f_g = f'final_A_cp_{n}.csv', f'final_A_gate_{n}.csv'
    if os.path.exists(f_cp) and os.path.exists(f_g):
        print(n, 'already done, skipping'); continue
    a, b = H.part_a(n, p)
    pd.DataFrame(a, columns=H.CP_COLS).to_csv(f_cp, index=False)
    pd.DataFrame(b, columns=H.GATE_COLS).to_csv(f_g, index=False)
    print(n, 'done after', round((time.time() - t0) / 60, 1), 'min')

## Cell 5: part B, 10 splits (forest and the three networks, class-by-class sets, attribute-level fidelity). About 30 to 60 minutes. Ignore any convergence warnings

In [ ]:
import warnings; warnings.filterwarnings('ignore')
t0 = time.time()
for n, p in paths.items():
    f1, f2 = f'final_B1_{n}.csv', f'final_B2_{n}.csv'
    if os.path.exists(f1) and os.path.exists(f2):
        print(n, 'already done, skipping'); continue
    a, b = H.part_b(n, p)
    pd.DataFrame(a, columns=H.B1_COLS).to_csv(f1, index=False)
    pd.DataFrame(b, columns=H.B2_COLS).to_csv(f2, index=False)
    print(n, 'done after', round((time.time() - t0) / 60, 1), 'min')

## Cell 6: look at the results

In [ ]:
pd.set_option('display.width', 220)
cp = pd.concat([pd.read_csv(f'final_A_cp_{n}.csv') for n in paths])
print(cp[cp.alpha == 0.10].groupby(['dataset', 'score', 'variant'])[H.METRIC_COLS].mean().round(3).to_string())
b1 = pd.concat([pd.read_csv(f'final_B1_{n}.csv') for n in paths])
print(b1[b1.variant == 'pooled'].groupby(['dataset', 'model'])[['accuracy', 'bal_acc', 'coverage', 'answered', 'fidelity']].mean().round(3).to_string())
b2 = pd.concat([pd.read_csv(f'final_B2_{n}.csv') for n in paths])
print(b2.groupby(['dataset', 'model'])[['share_defined', 'top1_agree', 'random_top1', 'spearman']].mean().round(3).to_string())

## Cell 7: download everything and send it to me

In [ ]:
!zip -q extra_results.zip final_A_cp_*.csv final_A_gate_*.csv final_B1_*.csv final_B2_*.csv
from google.colab import files
files.download('extra_results.zip')